<a href="https://colab.research.google.com/github/suneel-ratnala/Agentic_AI_Practice/blob/main/day1/lca-langchainV1-essentials/mandatory/L6_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="https://github.com/jayyanar/agentic-ai-training/blob/lab-day-1/batch2/lca-langchainV1-essentials/assets/LC_Memory_before.png?raw=1" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

What we're doing: Install required packages for the memory lesson in Colab.

In [1]:
!pip install -qU langchain-groq langgraph langchain-community pysqlite3-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
langchain 1.2.10 requires langgraph<1.1.0,>=1.0.8, but you have langgraph 1.1.2 which is incompatib

What we're doing: Load the GROQ API key from Colab userdata into the environment.

In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

What we're doing: Connect to the sample SQLite database and print available tables.

In [4]:
from langchain_community.utilities import SQLDatabase

# Download the database file locally
!wget -q https://github.com/jayyanar/agentic-ai-training/raw/lab-day-1/batch2/lca-langchainV1-essentials/Chinook.db

# Update the URI to the local path
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

# Verify it works
print(db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


What we're doing: Define `RuntimeContext` dataclass to hold the database reference for runtime use.

In [5]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

What we're doing: Define `execute_sql` tool to allow the agent to run read-only SQL queries via the runtime.

In [6]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

What we're doing: Set `SYSTEM_PROMPT` rules to constrain agent SQL behavior and ensure read-only queries.

In [7]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

What we're doing: Initialize the Groq model and create an agent with `execute_sql` and runtime context.

In [8]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

# Initialize the Groq model
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_retries=2,
)

agent = create_agent(
    #model="openai:gpt-5",
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

What we're doing: Stream a repeated query scenario to observe agent behavior without memory enabled.

In [9]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_a9ab9ef4-0cdd-418e-b612-75d7de696bb9)
 Call ID: fc_a9ab9ef4-0cdd-418e-b612-75d7de696bb9
  Args:
    query: SELECT name, sql FROM sqlite_master WHERE type='table';
================================= Tool Message =================================
Name: execute_sql

[('Album', 'CREATE TABLE [Album]\n(\n    [AlbumId] INTEGER  NOT NULL,\n    [Title] NVARCHAR(160)  NOT NULL,\n    [ArtistId] INTEGER  NOT NULL,\n    CONSTRAINT [PK_Album] PRIMARY KEY  ([AlbumId]),\n    FOREIGN KEY ([ArtistId]) REFERENCES [Artist] ([ArtistId]) \n\t\tON DELETE NO ACTION ON UPDATE NO ACTION\n)'), ('Artist', 'CREATE TABLE [Artist]\n(\n    [ArtistId] INTEGER  NOT NULL,\n    [Name] NVARCHAR(120),\n    CONSTRAINT [PK_Artist] PRIMARY KEY  ([ArtistId])\n)'), ('Customer', 

What we're doing: Follow up query without memory to show that prior context isn't retained across invocations.

In [15]:
#question = "When is my name?"

question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_cd4de27b-29b8-465f-bb71-880df698b63a)
 Call ID: fc_cd4de27b-29b8-465f-bb71-880df698b63a
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_2b32c74c-d921-4ba8-b323-4abde00dd487)
 Call ID: fc_2b32c74c-d921-4ba8-b323-4abde00dd487
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), 

## Add memory

What we're doing: Import an in-memory checkpointer to enable simple agent memory persistence.

In [16]:
from langgraph.checkpoint.memory import InMemorySaver

What we're doing: Re-create the agent with an in-memory checkpointer so it can remember prior interactions.

In [17]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    #model="openai:gpt-5",
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

What we're doing: Run the same invoice query with memory enabled (thread_id=1) to persist context between calls.

In [18]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_7fee1ce4-8dad-4617-89bb-3c17c2791576)
 Call ID: fc_7fee1ce4-8dad-4617-89bb-3c17c2791576
  Args:
    query: SELECT name, sql FROM sqlite_master WHERE type='table';
================================= Tool Message =================================
Name: execute_sql

[('Album', 'CREATE TABLE [Album]\n(\n    [AlbumId] INTEGER  NOT NULL,\n    [Title] NVARCHAR(160)  NOT NULL,\n    [ArtistId] INTEGER  NOT NULL,\n    CONSTRAINT [PK_Album] PRIMARY KEY  ([AlbumId]),\n    FOREIGN KEY ([ArtistId]) REFERENCES [Artist] ([ArtistId]) \n\t\tON DELETE NO ACTION ON UPDATE NO ACTION\n)'), ('Artist', 'CREATE TABLE [Artist]\n(\n    [ArtistId] INTEGER  NOT NULL,\n    [Name] NVARCHAR(120),\n    CONSTRAINT [PK_Artist] PRIMARY KEY  ([ArtistId])\n)'), ('Customer', 

What we're doing: Ask a follow-up question using the same memory `thread_id` to check recall of details like titles.

In [19]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_092cdf77-ef55-4fd1-86fd-16345974ce97)
 Call ID: fc_092cdf77-ef55-4fd1-86fd-16345974ce97
  Args:
    query: SELECT DISTINCT t.Name FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId WHERE il.InvoiceId = 374
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou',), ('Through The Never',), ('My Friend Of Misery',), ('The Wait',), ('Blitzkrieg',), ('So What',)]
================================== Ai Message ==================================

Here are the track titles that appear on your last invoice (Invoice #374):

1. Holier Than Thou  
2. Through The Never  
3. My Friend Of Misery  
4. The Wait  
5. Blitzkrieg  
6. So What  

These are the six songs included on that invoice.


## Try your own queries
Now that there is memory, check the agents recall!

What we're doing: Placeholder for you to enter your own question and test agent memory behavior.

In [20]:
question = "what was my name given earlier?"

#question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

what was my name given earlier?
================================== Ai Message ==================================

Your name, as provided earlier, is **Frank Harris**.


What we're doing: Example follow-up that asks the agent to show full results using the same memory thread.

In [21]:
question = "yes i want to see the full list"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

yes i want to see the full list
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_badb3e96-d05b-4f98-a13d-ec3ee7c7880f)
 Call ID: fc_badb3e96-d05b-4f98-a13d-ec3ee7c7880f
  Args:
    query: SELECT il.InvoiceLineId, t.Name, il.UnitPrice, il.Quantity FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId WHERE il.InvoiceId = 374
================================= Tool Message =================================
Name: execute_sql

[(2021, 'Holier Than Thou', 0.99, 1), (2022, 'Through The Never', 0.99, 1), (2023, 'My Friend Of Misery', 0.99, 1), (2024, 'The Wait', 0.99, 1), (2025, 'Blitzkrieg', 0.99, 1), (2026, 'So What', 0.99, 1)]
================================== Ai Message ==================================

Here’s the complete breakdown of every line item on your last invoice (Invoice #374):

| InvoiceLineId | Track Title          | Unit Price 

(End of notebook)